# Fundamentos de Optimización Lineal: SGD en Perceptrón, PA y SVM

**Autor:** Dr. José G. Fuentes  
**Instituto:** CADIT Universidad Anáhuac

---

## 1. Contexto Matemático: Dinámica del Hiperplano en $\mathbb{R}^d$

La familia de clasificadores lineales se abstrae bajo la función de decisión $f(x) = \text{sign}(\langle w, x \rangle + b)$. La distinción fundamental entre los algoritmos radica en la **función de pérdida empírica (Loss Function)** que intentan minimizar mediante Gradiente Estocástico (SGD):

1. **Perceptrón de Rosenblatt:** Minimiza el error de clasificación directo $L_P(w, (x,y)) = \max(0, -y \langle w, x \rangle)$. La actualización ocurre *únicamente* ante fallas de clasificación, careciendo de noción sistémica de margen de seguridad.
2. **Pasivo-Agresivo (PA-I):** Incorpora la noción de margen unitario acoplado a la pérdida Hinge $L_H(w, (x,y)) = \max(0, 1 - y \langle w, x \rangle)$. Su actualización es analítica, resolviendo un problema de optimización en forma cerrada con restricción de cercanía en cada iteración (agresividad paramétrica acotada por $C$).
3. **Máquina de Vector Soporte (SVM Primal):** Optimiza sistémicamente la pérdida Hinge regularizada vía norma L2: $L_{SVM}(w, (x,y)) = \frac{\lambda}{2} \|w\|^2 + \max(0, 1 - y \langle w, x \rangle)$. Genera estructuralmente un hiperplano de **margen máximo** altamente robusto al ruido.

> [!IMPORTANT]
> En esta implementación, analizaremos empíricamente la evolución lenta del gradiente de estos tres modelos en paralelo. Forzaremos **vectores de inicialización patológicos** y tasas de aprendizaje reducidas para permitir que el estudiante observe detalladamente la convergencia paramétrica (espacio de pesos) de forma simultánea a su reflejo en el decaimiento de la pérdida promedio.

## 2. Generación del Espacio Vectorial Topológico

Construiremos conjuntos linealmente separables pero con un factor de dispersión diseñado para conformar una **banda de separación estrecha**. Esto desafía el régimen de convergencia temprana y resalta las sutiles diferencias comportamentales entre los algoritmos al perfilar iterativamente la región del margen.

In [31]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
import ipywidgets as widgets
from IPython.display import display, clear_output

# Perfilado estético institucional riguroso (Anáhuac Orange)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['#FF6600', '#2B2B2B', '#4D4D4D'])
plt.rcParams['font.family'] = 'sans-serif'

def build_hard_separable_manifold(n_samples=5000):
    # Semilla determinista. Margen forzado a ser muy reducido para convergencia lenta.
    np.random.seed(42)
    X, y = make_blobs(n_samples=n_samples, centers=[[1.2, 1.2], [-1.2, -1.2]], 
                      n_features=2, cluster_std=2)
    # Mapeo proyectivo de etiquetas a la vecindad hiperbólica {-1, 1}
    y = np.where(y == 0, -1, 1)
    return X, y

X, y = build_hard_separable_manifold()


## 3. Motor Polimórfico de Descenso Estocástico (SGD)

Construimos una arquitectura de actualización capaz de mutar y seguir la traza estocástica de los tres modelos en un paso unificado. El seguimiento de la pérdida empírica acumulada es vital para verificar la monotonicidad general (aunque estocástica localmente) del descenso.

In [32]:
class StructuralLinearSGD:
    def __init__(self, mode, eta=0.005, C=0.02, lambda_param=0.1):
        self.mode = mode
        self.eta = eta
        self.C = C
        self.lambda_param = lambda_param
        
        # INICIALIZACIÓN PATOLÓGICA: 
        # Empezamos con un peso completamente ortogonal a la dirección correcta
        # para obligar al algoritmo a evidenciar el giro del hiperplano.
        self.w = np.array([-1.0, 1.0])  
        self.b = 0.0
        
        # Hooks de trazabilidad métrica
        self.history = []
        self.loss_history = []
        self.cumulative_loss = 0.0
        
    def _forward_pass(self, x, target):
        margin_algebraico = target * (np.dot(self.w, x) + self.b)
        loss = 0.0
        
        if self.mode == 'Perceptrón':
            loss = max(0.0, -margin_algebraico)
        elif self.mode == 'Pasivo-Agresivo':
            loss = max(0.0, 1.0 - margin_algebraico)
        elif self.mode == 'SVM Primal':
            loss = max(0.0, 1.0 - margin_algebraico) + 0.5 * self.lambda_param * np.linalg.norm(self.w)**2
            
        return loss, margin_algebraico

    def train(self, X, y, epochs=1):
        step = 0
        for epoch in range(epochs):
            # Aleatorización determinista para coherencia comparativa
            idx = np.random.RandomState(epoch).permutation(len(y))
            for i in idx:
                xi, yi = X[i], y[i]
                loss, margin = self._forward_pass(xi, yi)
                
                # Actualización de Tensor Subgradiente según Arquitectura
                if self.mode == 'Perceptrón':
                    if margin <= 0:
                        self.w = self.w + self.eta * yi * xi
                        self.b = self.b + self.eta * yi
                        
                elif self.mode == 'Pasivo-Agresivo':
                    if loss > 0:
                        tau = min(self.C, loss / (np.linalg.norm(xi)**2 + 1e-8))
                        self.w = self.w + tau * yi * xi
                        self.b = self.b + tau * yi
                        
                elif self.mode == 'SVM Primal':
                    if margin < 1.0:
                        self.w = self.w - self.eta * (self.lambda_param * self.w - yi * xi)
                        self.b = self.b + self.eta * yi
                    else:
                        self.w = self.w - self.eta * (self.lambda_param * self.w)
                        
                # Registro de estado microscópico
                self.history.append((self.w.copy(), self.b))
                self.cumulative_loss += loss
                self.loss_history.append(self.cumulative_loss / (step + 1)) # EMA simulada
                step += 1

# Orquestación Paralela de las 3 Arquitecturas
models = {
    'Perceptrón': StructuralLinearSGD(mode='Perceptrón', eta=0.1),
    'Pasivo-Agresivo': StructuralLinearSGD(mode='Pasivo-Agresivo', C=0.1),
    'SVM Primal': StructuralLinearSGD(mode='SVM Primal', eta=0.1, lambda_param=1)
}

for name, model in models.items():
    model.train(X, y, epochs=1)  # 250 iteraciones finas microscópicas


## 4. Análisis Trazable en Espejo: Hiperplano vs Pérdida ($L$)

El siguiente marco interactivo es el corazón analítico de la sesión. Proveemos un renderizado síncrono que vincula:
1. La dinámica geométrica y rotacional en el espacio dimensional de atributos (Izquierda).
2. El decaimiento estocástico riguroso del paisaje de la función de coste abstracto (Derecha).
Seleccione independientemente la arquitectura para observar la brutal diferencia, por ejemplo, entre la corrección ortogonal del Perceptrón y el arrastre de margen regulado de SVM.

In [33]:
def telemetry_dashboard(modelo, t):
    model = models[modelo]
    w, b = model.history[t]
    
    fig, (ax_geom, ax_loss) = plt.subplots(1, 2, figsize=(16, 6.5), gridspec_kw={'width_ratios': [1, 1.2]})
    fig.suptitle(f"Traza Estocástica: {modelo} | Iteración Topológica {t+1}", fontsize=15, fontweight='bold')
    
    # PANEL IZQUIERDO: Geometría Tensor/Hiperplano
    ax_geom.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k', alpha=0.5, s=35)
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    
    if abs(w[1]) > 1e-9:
        x_vals = np.array([x_min, x_max])
        y_vals = -(w[0] * x_vals + b) / w[1]
        ax_geom.plot(x_vals, y_vals, '-', color='#FF6600', lw=2.5, label='Hiperplano Activo')
        
        # Bandas de Confianza (Margen Funcional) dependientes de L2 Norm
        if modelo != 'Perceptrón':
            margin_geom = 1.0 / (np.linalg.norm(w) + 1e-9)
            norm_y = np.sqrt(1 + (w[0]/w[1])**2)
            ax_geom.plot(x_vals, y_vals + margin_geom * norm_y, '--', color='#2B2B2B', alpha=0.7, lw=1.2, label='Frontera Margen')
            ax_geom.plot(x_vals, y_vals - margin_geom * norm_y, '--', color='#2B2B2B', alpha=0.7, lw=1.2)

    ax_geom.set_xlim(x_min, x_max)
    ax_geom.set_ylim(X[:, 1].min() - 1, X[:, 1].max() + 1)
    ax_geom.set_title("Evolución Posicional en $\mathbb{R}^2$", pad=12)
    ax_geom.grid(True, linestyle=':', alpha=0.6)
    ax_geom.legend(loc='lower right')

    # PANEL DERECHO: Proyección de Pérdida en Espejo
    x_hist = range(t + 1)
    y_hist = model.loss_history[:t + 1]
    
    ax_loss.plot(x_hist, y_hist, color='#FF6600', lw=2)
    ax_loss.fill_between(x_hist, y_hist, alpha=0.1, color='#FF6600')
    ax_loss.scatter([t], [y_hist[-1]], color='#2B2B2B', s=90, zorder=5) # Foco en iteración t
    
    ax_loss.set_xlim(0, len(model.history))
    max_loss = max(model.loss_history)
    ax_loss.set_ylim(0.0, (max_loss * 1.1) if max_loss > 0 else 1.0)
    
    ax_loss.set_title(f"Decaimiento EMA de $L$ Subgradiente", pad=12)
    ax_loss.set_xlabel("Paso Estocástico ($t$)")
    ax_loss.set_ylabel("$\mathbb{E}[\mathcal{L}(w_t)]$")
    ax_loss.grid(True, linestyle=':', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

# Conector de Eventos y Renderizado Interactivo
alg_selector = widgets.Dropdown(options=list(models.keys()), value='SVM Primal', description='Algoritmo:', style={'description_width': 'initial'})
time_slider = widgets.IntSlider(min=0, max=4999, step=1, value=0, description='Micro-Iteración ($t$):', layout=widgets.Layout(width='650px'), style={'description_width': 'initial'})

widgets.interactive(telemetry_dashboard, modelo=alg_selector, t=time_slider)


<>:26: SyntaxWarning: invalid escape sequence '\m'
<>:44: SyntaxWarning: invalid escape sequence '\m'
<>:26: SyntaxWarning: invalid escape sequence '\m'
<>:44: SyntaxWarning: invalid escape sequence '\m'
/var/folders/b2/142kycp96wdg69k6pkw9_vf00000gp/T/ipykernel_36728/278628276.py:26: SyntaxWarning: invalid escape sequence '\m'
  ax_geom.set_title("Evolución Posicional en $\mathbb{R}^2$", pad=12)
/var/folders/b2/142kycp96wdg69k6pkw9_vf00000gp/T/ipykernel_36728/278628276.py:44: SyntaxWarning: invalid escape sequence '\m'
  ax_loss.set_ylabel("$\mathbb{E}[\mathcal{L}(w_t)]$")


interactive(children=(Dropdown(description='Algoritmo:', index=2, options=('Perceptrón', 'Pasivo-Agresivo', 'S…